# 分钟动作价值期限审计

仅复用冻结现金买SOXX动作，在预先固定的09:48、12:00和15:30三个分钟，比较同日收盘、下一开盘和第5会话下一开盘的自身状态费后增量。此卡无新信号、无拟合、无候选账户；不根据结果选时点或期限。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_episode import replay_event_full_path
from src.backtest.stage1_event_checkpoint import resume_episode
from src.backtest.stage1_event_checkpoint_batch import iter_after_open_checkpoints
from src.backtest.ibkr_tiered import settle_tiered_event
from src.backtest.event_guard_clock import guard_slot
from copy import deepcopy
import exchange_calendars as xc
from qlib.data import D
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
root=Path.cwd();card=root/"research/stage1_minute_value_horizon_audit_20261029"
raw=(card/"plan.json").read_bytes();plan=json.loads(raw);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan["replay_code"])==plan["replay_code_sha256"]
assert sha(root/plan["batch_code"])==plan["batch_code_sha256"]
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"] and sha(old/"plan.json")==plan["old_source_plan_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
minute_plan=json.loads((root/plan["minute_source_plan"]).read_text())
assert sha(root/minute_plan["provider_uri"]/"qlib_store_manifest.json")==minute_plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/minute_plan["provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{year}" for year in parent_plan["signal_years"]],*[f"models/recovery_{year}" for year in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and len(book["authority"])==len(frame) and book["audit"]["status"]=="passed"
open_models={year:DatedModel(objects[f"models/open_{year}"],f"{year-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{year}") for year in parent_plan["signal_years"]}
recovery_models={}
for year in parent_plan["signal_years"]:
    if year in parent_plan["explicit_no_recovery_model_years"]:
        recovery_models[year]=DatedModel(None,f"{year-1}-12-31",f"explicit_no_model_{year}_insufficient_prior_teacher",explicit_no_model=True)
    else:
        recovery_models[year]=DatedModel(objects[f"models/recovery_{year}"],f"{year-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{year}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])

old_action_plan=json.loads((root/plan["fixed_action_source_plan"]).read_text())
assert old_action_plan["action"]["type"]=="direct_cash_to_soxx_with_original_beta_headroom"


In [2]:

# Read the full Qlib minute plane once; event clocks are selected without future returns.
from qlib.data import D
import exchange_calendars as xc
bars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],
    start_time="2018-10-15 09:31:00",end_time="2022-12-22 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
    item=bars.xs(symbol,level="instrument").copy()
    item.columns=[symbol+"_"+field.removeprefix("$") for field in item]
    parts.append(item)
quotes=pd.concat(parts,axis=1).sort_index()
schedule=xc.get_calendar("XNYS",start=frame.trade_date.min(),end=frame.trade_date.max()).schedule
closes={pd.Timestamp(d).tz_localize(None):pd.Timestamp(r.close).tz_convert("America/New_York").tz_localize(None) for d,r in schedule.iterrows()}
assert len(frame)==len(book["independent_replay"])


In [3]:

records=[];counts={"source_invalid":0,"recovery":0,"terminal":0,"no_funded":0,"funded":0}
for checkpoint in iter_after_open_checkpoints(frame,controller_by_date=controller,
        recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],plan=parent_plan):
    i=checkpoint.index;row=frame.iloc[i];day=pd.Timestamp(row.trade_date)
    if i+5>len(frame):counts["terminal"]+=1;continue
    if bool(book["authority"].iloc[i].recovery_action):counts["recovery"]+=1;continue
    try:asset=quotes.loc[day.strftime("%Y-%m-%d")]
    except KeyError:counts["source_invalid"]+=1;continue
    if asset.empty or not asset[["SOXL_price_valid","SOXX_price_valid"]].eq(1).all().all():
        counts["source_invalid"]+=1;continue
    close=closes[day]
    hold=resume_episode(frame,checkpoint,controller_by_date=controller,recovery_spec=recovery_spec,
        action_spec=action_spec,selection=prepared["selection"],plan=parent_plan,event=None,horizon_sessions=5)
    pd.testing.assert_frame_equal(hold["replay"].reset_index(drop=True),
        book["independent_replay"].iloc[i:i+5].reset_index(drop=True),
        check_dtype=False,rtol=0,atol=1e-7)
    for hhmm in plan["completed_bar_times_et"]:
        stamp=pd.Timestamp(str(day.date())+" "+hhmm+":00")
        if stamp not in asset.index:counts["no_funded"]+=1;continue
        slot=guard_slot(stamp,session_open=day+pd.Timedelta(hours=9,minutes=30),
            session_close=close,original_open_fill=day+pd.Timedelta(hours=9,minutes=47))
        if slot is None or slot.fill_at+pd.Timedelta(minutes=1) not in asset.index:
            counts["no_funded"]+=1;continue
        bar=asset.loc[stamp];mark=np.array([bar.SOXL_close,bar.SOXX_close],float)
        nav=float(checkpoint.state.q@mark+checkpoint.state.cash)
        w=np.r_[checkpoint.state.q*mark,checkpoint.state.cash]/nav
        cap=float(checkpoint.opened["current_beta_cap"])
        d=min(.10,max(0.,w[2]-.15),max(0.,cap-(3*w[0]+w[1])),max(0.,1-w[1]))
        if d<=1e-10:counts["no_funded"]+=1;continue
        target=w+np.array([0.,d,-d])
        fill_bar=asset.loc[slot.fill_at+pd.Timedelta(minutes=1)]
        adjusted=np.array([fill_bar.SOXL_open,fill_bar.SOXX_open],float)
        raw=np.array([fill_bar.SOXL_raw_open,fill_bar.SOXX_raw_open],float)
        traded=settle_tiered_event(checkpoint.state.q,checkpoint.state.cash,desired=target,
            adjusted_prices=adjusted,raw_prices=raw,stamp=slot.fill_at,fees=deepcopy(checkpoint.state.fees))
        q,cash=traded["quantities"],float(traded["cash"])
        fill_nav=float(q@adjusted+cash)
        legal=fill_nav>0 and cash/fill_nav>=.15-1e-8 and 3*q[0]*adjusted[0]/fill_nav+q[1]*adjusted[1]/fill_nav<=cap+.001+1e-8
        orders=traded["orders"]
        if not(legal and len(orders)==1 and orders[0]["symbol"]=="SOXX" and not orders[0]["sell"]):
            counts["no_funded"]+=1;continue
        event={"trade_date":day,"bar_end_at":stamp,"decision_at":slot.decision_at,
            "fill_at":slot.fill_at,"desired_weights":target,"adjusted_fill_prices":adjusted,
            "raw_fill_prices":raw}
        branch=resume_episode(frame,checkpoint,controller_by_date=controller,
            recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],
            plan=parent_plan,event=event,horizon_sessions=5)
        assert len(branch["orders"].query('slot == "event"'))==1
        same_day=(float(q@row[["close_soxl","close_soxx"]].to_numpy(float)+cash)-
                  float(checkpoint.state.q@row[["close_soxl","close_soxx"]].to_numpy(float)+checkpoint.state.cash))/nav
        next_open=(float(branch["replay"].iloc[0].native_expected_nav)-float(hold["replay"].iloc[0].native_expected_nav))/nav
        fifth=(float(branch["replay"].iloc[4].native_expected_nav)-float(hold["replay"].iloc[4].native_expected_nav))/nav
        records.append({"trade_date":str(day.date()),"year":int(day.year),"bar_end_at":hhmm,
            "same_day_close":same_day,"next_open":next_open,"fifth_session_next_open":fifth,
            "delta_target":float(d),"event_raw_shares":int(orders[0]["raw_shares"])})
        counts["funded"]+=1
assert counts["funded"]==len(records) and len(records)<=plan["budget"]["event_slots"]
values=pd.DataFrame(records)
assert not values.empty
annual=[]
for (year,tm),part in values.groupby(["year","bar_end_at"]):
    annual.append({"year":int(year),"bar_end_at":tm,"days":len(part),
        **{h+"_mean":float(part[h].mean()) for h in plan["horizons"]},
        **{h+"_positive_fraction":float(part[h].gt(0).mean()) for h in plan["horizons"]},
        "next_open_to_fifth_sign_changed":int((np.sign(part.next_open)!=np.sign(part.fifth_session_next_open)).sum())})
annual=pd.DataFrame(annual).sort_values(["year","bar_end_at"])
(card/"annual_horizon_summary.csv").write_text(annual.to_csv(index=False))
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),
    "decision":"descriptive_horizon_audit_only_no_alpha_admission",
    "parent_recorder_id":plan["parent_recorder_id"],"source_readback_entries":len(receipt),
    "counts":counts,"funded_days":int(values.trade_date.nunique()),"annual_funded_days":{str(y):int(v.trade_date.nunique()) for y,v in values.groupby("year")},"annual_rows":len(annual),
    "overall":{h:{"mean":float(values[h].mean()),"positive_fraction":float(values[h].gt(0).mean())} for h in plan["horizons"]},
    "new_model_fits":0,"candidate_accounts":0,"historical_research_passed":False,
    "source_ever_first_receipt":"unknown","dividend_receivable_parity":"unknown"}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(audit,ensure_ascii=False,indent=2))


## 中文结论

在已冻结的09:48、12:00、15:30时点，538日产生996个合法整股事件；HOLD五会话父本逐行对账。固定现金买SOXX动作相对HOLD的同日收盘、下一开盘、第5会话下一开盘平均费后增量分别为−0.068、+0.208、+0.593bp。2022年三个时点的第5会话均值全负；2020全正。此为描述性已复用历史，不是分钟信号或完整策略超额；无模型拟合、无候选账户。历史首收及单独分红应收未知，原失败信息版本保持关闭。逐年见annual_horizon_summary.csv，实际运行及来源见audit.json。下一步仅在新经济机制与同状态控制事前冻结后筛信息，不按本卡回选时点或期限。